# Bibliothèque complète des fiches CIM-10

Génération massive des fiches markdown pour tous les codes feuilles du CSV.

Architecture : la logique métier vit dans `src/recode_icd/cards.py` (factorisée depuis l'ancien prototype). Ce notebook est l'outil interactif pour mise au point, mesure de performance et inspection ad-hoc.

Équivalents non-interactifs : `uv run recode-icd cards build` ou `uv run python scripts/build_cards_library.py`.

In [ ]:
from __future__ import annotations

import logging
import random
import time
from pathlib import Path

import polars as pl

from recode_icd.cards import DEFAULT_SEED, build_card, build_cards_library
from recode_icd.utils.loaders_dev import load_exploration_context

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

ctx = load_exploration_context(with_external=True)
assert ctx.flat is not None and ctx.merged is not None

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(30)

## 1 — Inventaire des codes

Combien de codes uniques dans le CSV, distribution par chapitre.

In [ ]:
csv = ctx.flat
assert isinstance(csv, pl.DataFrame)
codes = csv["code"].unique().to_list()
print(f"Codes uniques dans le CSV : {len(codes):,}")

merged = ctx.merged
assert isinstance(merged, pl.DataFrame)
by_chap = (
    pl.DataFrame({"code": codes})
    .join(
        merged.with_columns(
            pl.col("path").str.split("/").list.get(0).alias("chap")
        ).select("code", "chap"),
        on="code", how="left",
    )
    .group_by("chap").len().sort("len", descending=True)
)
print("\nDistribution par chapitre :")
print(by_chap)

## 2 — Génération d'un petit échantillon (10 codes)

Spot-check du rendu sur quelques codes témoins.

In [ ]:
rng = random.Random(DEFAULT_SEED)
sample_codes = ["A18.1", "J18.8", "R51", "U07.1", "M01.08", "M01.05", "M00.00"]
for code in sample_codes:
    card = build_card(code, ctx, rng)
    print(f"--- {code} ({len(card)} chars) ---")
    print(card[:200] + "...\n" if len(card) > 200 else card + "\n")

## 3 — Mesure de performance

Sur 50 codes échantillonnés au hasard, mesurer le temps moyen et projeter sur la bibliothèque complète.

In [ ]:
_r = random.Random(0)
perf_sample = _r.sample(codes, 50)
rng_perf = random.Random(DEFAULT_SEED)

t0 = time.perf_counter()
for c in perf_sample:
    _ = build_card(c, ctx, rng_perf)
elapsed = time.perf_counter() - t0
per_card_ms = (elapsed / 50) * 1000
projection_min = (per_card_ms * len(codes)) / 1000 / 60

print(f"50 fiches en {elapsed:.2f}s → {per_card_ms:.1f} ms/fiche")
print(f"Projection {len(codes):,} codes : {projection_min:.1f} min")

## 4 — Génération massive

`build_cards_library` génère toutes les fiches sous `outputs/cards_library/<chapitre>/<code>.md` et produit `_index.csv`. Log de progression tous les 1 000 codes.

In [ ]:
summary = build_cards_library(
    ctx=ctx,
    output_dir=Path("outputs/cards_library"),
    progress=True,
)
print(f"\nCodes total       : {summary.n_codes_total:,}")
print(f"Fiches écrites    : {summary.n_written:,}")
print(f"Erreurs           : {summary.n_errors}")
print(f"Durée             : {summary.elapsed_seconds:.1f} s")
print(f"Index             : {summary.index_path}")

## 5 — Vérifications post-génération via `_index.csv`

Inventaire par chapitre, distribution `nb_chars`, présence des 4 sections.

In [ ]:
index = pl.read_csv(summary.index_path)
print(f"Lignes dans _index.csv : {index.height:,}\n")

print("Distribution par chapitre :")
print(index.group_by("chapter").len().sort("len", descending=True))

print("\nPrésence des sections :")
for col in ("has_perimetre", "has_localisations", "has_exclusions", "has_formulations"):
    n = index.filter(pl.col(col)).height
    print(f"  {col:25s} : {n:5,} / {index.height:5,}  ({100*n/index.height:.1f}%)")

print("\nStats nb_chars :")
print(
    index.select(
        pl.col("nb_chars").min().alias("min"),
        pl.col("nb_chars").quantile(0.25).alias("q25"),
        pl.col("nb_chars").median().alias("med"),
        pl.col("nb_chars").quantile(0.75).alias("q75"),
        pl.col("nb_chars").max().alias("max"),
        pl.col("nb_chars").mean().alias("mean"),
        pl.col("nb_chars").sum().alias("total"),
    )
)

## 6 — Spot-check sur 5 fiches au hasard

In [ ]:
_r2 = random.Random(123)
spot_sample = _r2.sample(codes, 5)
for c in spot_sample:
    fp = summary.output_dir / index.filter(pl.col("code") == c)["filepath"][0]
    print(f"=== {c} ({fp}) ===")
    print(fp.read_text()[:600])
    print("...\n")